# Heat Exchanger v3 - Telemetry + Labels Generation


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm


In [2]:

class HeatExchangerTwin:

    def __init__(self, asset_id, failure_scenario):

        self.asset_id = asset_id
        self.failure_scenario = failure_scenario

        self.health = 100.0
        self.fouling_index = 0
        self.corrosion_index = 0
        self.tube_leakage_index = 0

    def update_health(self):

        wear = np.random.uniform(0.00015, 0.0008)

        if self.failure_scenario == "Fouling":
            self.fouling_index += wear * 5
            self.health -= wear * 1.8

        elif self.failure_scenario == "Corrosion":
            self.corrosion_index += wear * 5
            self.health -= wear * 1.5

        elif self.failure_scenario == "Tube Leakage":
            self.tube_leakage_index += wear * 5
            self.health -= wear * 1.4

        else:
            self.health -= wear * 0.3

        self.health = max(self.health, 0)

    def get_rul(self):
        return round((self.health / 100) * 240, 2)

    def get_failure_next_30_days(self):
        return int(self.get_rul() <= 30)

    def generate_telemetry(self):

        self.update_health()

        hot_inlet_temp = 280 + np.random.normal(0,2)

        hot_outlet_temp = (
            210 +
            self.fouling_index * 4 +
            np.random.normal(0,2)
        )

        cold_inlet_temp = 50 + np.random.normal(0,1)

        cold_outlet_temp = (
            130 -
            self.fouling_index * 3 +
            np.random.normal(0,1)
        )

        pressure_drop = (
            2 +
            self.fouling_index * 0.5 +
            np.random.normal(0,0.05)
        )

        heat_transfer_efficiency = max(
            50,
            95 - self.fouling_index * 2
        )

        telemetry = {
            "asset_id": self.asset_id,
            "hot_inlet_temp": round(hot_inlet_temp,2),
            "hot_outlet_temp": round(hot_outlet_temp,2),
            "cold_inlet_temp": round(cold_inlet_temp,2),
            "cold_outlet_temp": round(cold_outlet_temp,2),
            "pressure_drop": round(pressure_drop,2),
            "heat_transfer_efficiency": round(heat_transfer_efficiency,2),
            "fouling_index": round(self.fouling_index,4),
            "corrosion_index": round(self.corrosion_index,4)
        }

        labels = {
            "asset_id": self.asset_id,
            "failure_mode": self.failure_scenario,
            "rul_days": self.get_rul(),
            "failure_next_30_days": self.get_failure_next_30_days()
        }

        return telemetry, labels


In [3]:

assets = [
    HeatExchangerTwin("E-301","Fouling"),
    HeatExchangerTwin("E-302","Corrosion"),
    HeatExchangerTwin("E-303","Tube Leakage"),
    HeatExchangerTwin("E-304","Fouling"),
    HeatExchangerTwin("E-305","Healthy")
]


In [4]:

telemetry_records = []
label_records = []

minutes = 90 * 24 * 60
start_time = datetime.now()

for minute in tqdm(range(minutes)):

    current_time = start_time + timedelta(minutes=minute)

    for asset in assets:

        telemetry, labels = asset.generate_telemetry()

        telemetry["timestamp"] = current_time
        labels["timestamp"] = current_time

        telemetry_records.append(telemetry)
        label_records.append(labels)

telemetry_df = pd.DataFrame(telemetry_records)
labels_df = pd.DataFrame(label_records)

telemetry_df.to_csv("heat_exchanger_telemetry.csv", index=False)
labels_df.to_csv("heat_exchanger_labels.csv", index=False)

print(telemetry_df.shape)
print(labels_df.shape)


100%|██████████| 129600/129600 [00:12<00:00, 10697.80it/s]


(648000, 10)
(648000, 5)
